In [1]:
import pandas as pd
import os

## Transfer txt Files to csv files 


In [4]:
files_txt = "Datatxt"        
files_csv = "Datacsv" 

os.makedirs(files_csv, exist_ok=True)

In [6]:
for file in os.listdir(files_txt):
    if file.endswith(".txt"):
        file_path = os.path.join(files_txt, file)
        
        df = pd.read_csv(file_path, sep="$", low_memory=False)
        
        new_name = file.replace(".txt", ".csv")
        df.to_csv(os.path.join(files_csv, new_name), index=False)
        
        print(f"Converted: {file}")

Converted: DEMO25Q2.txt
Converted: DRUG25Q2.txt
Converted: INDI25Q2.txt
Converted: OUTC25Q2.txt
Converted: REAC25Q2.txt
Converted: RPSR25Q2.txt
Converted: THER25Q2.txt


## check It's work? 


In [10]:
demo = pd.read_csv(f"{files_csv}/DEMO25Q2.csv", low_memory=False)

df = demo.copy()
del demo
df.head()

,primaryid,caseid,caseversion,i_f_code,event_dt,mfr_dt,init_fda_dt,fda_dt,rept_cod,auth_num,...,age_grp,sex,e_sub,wt,wt_cod,rept_dt,to_mfr,occp_cod,reporter_country,occr_country
0,100236273,10023627,3,F,NaN,20250514,20140320,20250522,EXP,NaN,...,C,M,Y,NaN,NaN,20250522,NaN,LW,US,US
1,1015648244,10156482,44,F,20121101.0,20250603,20140507,20250605,EXP,NaN,...,E,F,Y,63.5,KG,20250605,NaN,CN,CA,CA
2,1016133066,10161330,66,F,NaN,20250529,20140509,20250602,EXP,NaN,...,E,F,Y,69.0,KG,20250602,NaN,CN,CA,CA
3,101815233,10181523,3,F,20140319.0,20250528,20140519,20250605,EXP,NaN,...,NaN,M,Y,89.0,KG,20250605,NaN,PH,GB,GB
4,102382176,10238217,6,F,NaN,20250414,20140616,20250422,EXP,NaN,...,T,M,Y,NaN,NaN,20250422,NaN,LW,US,US


## Function before merge to prevent duplicatis 

In [11]:
def prepare_table(file_name):
    df_temp = pd.read_csv(f"{files_csv}/{file_name}.csv", low_memory=False)
    
    df_temp = df_temp.groupby("primaryid").agg(
        lambda x: ", ".join(x.astype(str))
    ).reset_index()
    
    df_temp = df_temp.rename(columns=lambda x: x + "_" + file_name.lower() if x != "primaryid" else x)
    
    return df_temp

# Merge All CSVs Files Together 

In [1]:
import pandas as pd

tables = ["DEMO", "DRUG", "REAC", "THER", "OUTC", "RPSR","INDI"]
all_data = {}

for table in tables:
    dfs = []
    
    for q in ["Q1", "Q2", "Q3", "Q4"]:
        path = f"../Data/Datacsv/{table}25{q}.csv"
        df = pd.read_csv(path, low_memory=False)
        dfs.append(df)
    
    full_df = pd.concat(dfs, ignore_index=True)
    all_data[table] = full_df
    
    print(f"{table} shape:", full_df.shape)

DEMO shape: (1617444, 25)
DRUG shape: (7801018, 20)
REAC shape: (5657830, 4)
THER shape: (2002380, 7)
OUTC shape: (1232582, 3)
RPSR shape: (43939, 3)


In [ ]:
grouped_data = {}

for name in ["DRUG", "REAC", "THER", "OUTC", "RPSR","INDI"]:
    df = all_data[name]
    
    grouped = df.groupby("primaryid").agg(
        lambda x: ", ".join(x.astype(str))
    ).reset_index()
    
    grouped = grouped.rename(columns=lambda x: x + "_" + name.lower() if x != "primaryid" else x)
    
    grouped_data[name] = grouped
    
    print(f"{name} grouped done")

In [ ]:
final_df = all_data["DEMO"].copy()

for name in grouped_data:
    final_df = final_df.merge(grouped_data[name], on="primaryid", how="left")
    
    print(f"Merged {name}")

# Final Dataset 


In [ ]:
final_df.to_csv("2025_full_data.csv", index=False)